# 10 · Learned hybrid recommendation with LightFM

## Project context

This notebook follows the completed experiments in `07_hybrid_recommender.ipynb` and `09_reserved_slot_hybrid.ipynb`. Our current strongest reference is **implicit ALS alone**. The frozen **content-v1** representation uses MovieLens genres + release decade. **TMDB-v2** extends that base representation with TMDB genres, keywords, overview text, director, and cast.

The earlier experiments found:

- Score-percentile blends of ALS and content substantially reduced overall full-catalogue NDCG@10.
- Reserved-slot recommendations increased rare-movie exposure, but no tested candidate passed the project's 98% overall-NDCG retention floor.
- Profile-based content similarity did not retrieve any of the held-out strict-cold targets into the top 10 in the tested validation window.

We are now changing the learning architecture, not continuing to tune manual blending weights.

## What LightFM changes

LightFM learns latent embeddings for user/item identities and can also learn embeddings for item features. In the metadata-enhanced model, an item's representation is a weighted combination of its identity embedding and its metadata-feature embeddings. This lets interaction data teach the model which metadata patterns are useful for recommendation, rather than assuming that content similarity alone is the correct ranking signal.

Official references:

- [LightFM project and installation](https://github.com/lyst/lightfm)
- [LightFM quickstart](https://github.com/lyst/lightfm/blob/master/doc/quickstart.rst)
- [LightFM API documentation](https://making.lyst.com/lightfm/docs/lightfm.html)

## Candidates

1. **ALS-only** — existing collaborative reference, refitted on training data only.
2. **LightFM identity-only** — collaborative model with user/item identity features and no movie metadata.
3. **LightFM + content-v1** — item identity plus genres and release decade.
4. **LightFM + TMDB-v2** — item identity plus the saved TMDB-enriched content representation.

The identity-only LightFM comparison is important: it isolates whether adding metadata helps *within LightFM*, rather than confusing a change in algorithm with a benefit from the features.

## 1 · Experimental protocol and decision rules

### Fixed data and target definition

- Dataset: MovieLens `sample_10pct` and its existing train/validation split.
- Positive interaction / relevance: rating `>= 4.0`.
- LightFM loss: WARP, a pairwise ranking loss for implicit positive interactions.
- Catalogue: 17,241 modelled movies; item/user indices must align with `recutils.load_split`.
- Validation target window: positive validation interactions at or after `2016-01-01 00:00:00 UTC`, matching notebook 07/09.
- Tail pool: movies with fewer than 20 positive interactions in `train`.
- Strict-cold pool: zero positive training interactions; warm-tail pool: 1–19 positives.
- View A: all catalogue movies compete for the top 10; only target relevance is restricted when calculating tail/cold/warm-tail metrics.
- Seen training movies are never recommended.

### Controlled LightFM comparison

All LightFM candidates use the same initial hyperparameters: 32 components, WARP loss, learning rate `0.05`, `15` epochs, and fixed seed. These are a controlled first experiment, not a claim that the hyperparameters are optimal. We do not run a large grid in this notebook.

Metadata row weighting is fixed before looking at validation results: for movies with available features, identity features carry half of the item-feature weight and the normalized metadata group carries the other half. For movies with no metadata features, identity weight remains 1.0. This prevents the large number of text columns from winning merely because their raw weights sum to more than the identity feature.

### Decision rule

- Primary model-development question: does metadata-enhanced LightFM improve tail View A NDCG@10 versus identity-only LightFM? Report the paired user-level 95% bootstrap interval; a point-estimate increase alone is not enough evidence.
- Project-level viability guardrail: any candidate promoted over ALS must have overall View A NDCG@10 at least 98% of the ALS-only result calculated in this notebook.
- Among candidate models that pass the point-estimate guardrail, rank by tail View A NDCG@10. Report a paired confidence interval for the guardrail margin as well; a point-estimate pass is not formal proof of non-inferiority.
- Test data remains sealed. No test split is loaded into a variable, evaluated, or used to select settings.

### Important limitation

The split is an established temporal split, while the metric target window is filtered to 2016 onward to match the previous full-catalogue report. The cutoff aligns the evaluated targets; it does **not** prove that every training interaction globally precedes 2016.

## 2 · Environment setup

Run the notebook from the project's existing `.venv` / Python environment. LightFM is an additional dependency.

If the import below fails, stop and install it in the same environment used by this notebook:

```powershell
python -m pip install lightfm
```

If installation fails on Windows due to compilation/build dependencies, send the complete installation error rather than changing Python versions or installing random build tools. The LightFM project also documents Conda installation (`conda install -c conda-forge lightfm`).

This notebook uses only saved local feature matrices; **it makes no TMDB API calls**.

In [1]:
# Setup and dependency checks
import os
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import lightfm
    from lightfm import LightFM
except ImportError as exc:
    raise ImportError(
        "LightFM is not installed in this notebook's Python environment. "
        "In the project's active environment run: python -m pip install lightfm. "
        "If it fails to build on Windows, send the complete error output."
    ) from exc

# Find the project root, then use the same notebooks/ working directory convention
# used by the existing project notebooks and scripts/recutils.py.
_START = Path.cwd().resolve()
_PROJECT_ROOT = None
for _p in [_START, *_START.parents]:
    if (_p / "scripts" / "recutils.py").exists() and (_p / "data").exists():
        _PROJECT_ROOT = _p
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing scripts/recutils.py and data/. "
        "Place this notebook in the project's notebooks/ folder and reopen it there."
    )
_NOTEBOOK_DIR = _PROJECT_ROOT / "notebooks"
if _NOTEBOOK_DIR.exists() and Path.cwd().resolve() != _NOTEBOOK_DIR.resolve():
    os.chdir(_NOTEBOOK_DIR)
sys.path.insert(0, str(_PROJECT_ROOT / "scripts"))

from recutils import load_split

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42
CUTOFF_TS = int(pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp())
RETENTION_FRACTION = 0.98
N_BOOT = 1000
N_TIE_SEEDS = 10
SCORE_BATCH_SIZE = 64
NUM_THREADS = 1  # reproducibility first; increase only if runtime requires it

# First controlled LightFM configuration. Keep this fixed across the three models.
LIGHTFM_COMPONENTS = 32
LIGHTFM_EPOCHS = 15
LIGHTFM_LEARNING_RATE = 0.05
LIGHTFM_ITEM_ALPHA = 1e-6
LIGHTFM_USER_ALPHA = 1e-6
LIGHTFM_MAX_SAMPLED = 10

print("Project root:", _PROJECT_ROOT)
print("LightFM version:", getattr(lightfm, "__version__", "version unavailable"))
print("LightFM config:", {
    "loss": "warp",
    "no_components": LIGHTFM_COMPONENTS,
    "epochs": LIGHTFM_EPOCHS,
    "learning_rate": LIGHTFM_LEARNING_RATE,
    "item_alpha": LIGHTFM_ITEM_ALPHA,
    "user_alpha": LIGHTFM_USER_ALPHA,
    "max_sampled": LIGHTFM_MAX_SAMPLED,
    "num_threads": NUM_THREADS,
})

Project root: /mnt/c/Users/Lenovo/Documents/movie-recommender
LightFM version: 1.17
LightFM config: {'loss': 'warp', 'no_components': 32, 'epochs': 15, 'learning_rate': 0.05, 'item_alpha': 1e-06, 'user_alpha': 1e-06, 'max_sampled': 10, 'num_threads': 1}


## 3 · Load the established split and frozen feature artifacts

We load only `train` and `val` from the split object. The test split is deliberately not assigned or inspected. Feature matrices must be in the exact same internal item order as the split's `item_map`.

The validation target window is constructed once and reused for every model and every evaluation scope.

In [2]:
# Load train/validation data only; do not read S["test"].
S = load_split(NAME)
train = S["train"].copy()
val = S["val"].copy()
item_map = S["item_map"].sort_values("i").reset_index(drop=True)
user_map = S["user_map"].sort_values("u").reset_index(drop=True)
n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

feature_dir = _PROJECT_ROOT / "data" / "interim" / "hybrid_features"
tmdb_dir = _PROJECT_ROOT / "data" / "processed" / NAME
v1_path = feature_dir / "content_v1.npz"
tmdb_path = tmdb_dir / "tmdb_content_v2.npz"
tmdb_index_path = tmdb_dir / "tmdb_content_v2_index.parquet"

for _path in (v1_path, tmdb_path, tmdb_index_path):
    if not _path.exists():
        raise FileNotFoundError(
            f"Required local artifact not found: {_path}\n"
            "Generate it in the earlier content/TMDB notebooks first."
        )

X_v1 = sp.load_npz(v1_path).tocsr().astype(np.float32)
X_tmdb_v2 = sp.load_npz(tmdb_path).tocsr().astype(np.float32)
tmdb_item_index = pd.read_parquet(tmdb_index_path).sort_values("i").reset_index(drop=True)

assert len(train) > 0 and len(val) > 0
assert item_map["u"].is_unique if "u" in item_map.columns else True
assert item_map["i"].is_unique and item_map["movieId"].is_unique
assert user_map["u"].is_unique and user_map["userId"].is_unique
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert X_v1.shape == (n_items, 31), f"Unexpected content-v1 shape: {X_v1.shape}"
assert X_tmdb_v2.shape[0] == n_items, f"Unexpected TMDB-v2 shape: {X_tmdb_v2.shape}"
assert np.array_equal(tmdb_item_index["i"].to_numpy(), item_map["i"].to_numpy())
assert np.array_equal(tmdb_item_index["movieId"].to_numpy(), item_map["movieId"].to_numpy())
assert np.isfinite(X_v1.data).all() and np.isfinite(X_tmdb_v2.data).all()

# Seen mask includes ALL training ratings; only recommendations should be unseen.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(dtype=np.int64), train["i"].to_numpy(dtype=np.int64)),
    ),
    shape=(n_users, n_items),
)
seen_train.sum_duplicates()
seen_train.data[:] = 1.0

train_positive = train.loc[train["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
n_pos = np.bincount(train_positive["i"].to_numpy(dtype=np.int64), minlength=n_items)
tail_mask = n_pos < 20
strict_cold_mask = n_pos == 0
warm_tail_mask = (n_pos >= 1) & (n_pos < 20)
item_has_positive_history = n_pos > 0

# The established fallback is based on all training interactions, not only positive ones.
popularity_scores = np.bincount(
    train["i"].to_numpy(dtype=np.int64), minlength=n_items
).astype(np.float64)
u_has_pos = np.bincount(
    train_positive["u"].to_numpy(dtype=np.int64), minlength=n_users
) > 0

assert "timestamp" in val.columns, "Validation split lacks timestamp column."
val_ts = pd.to_numeric(val["timestamp"], errors="raise")
val_window = val.loc[
    (val["rating"] >= REL) & (val_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

def relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

item_idx_val = val_window["i"].to_numpy(dtype=np.int64)
relevant_by_scope = {
    "overall": relevant_dict(val_window),
    "tail": relevant_dict(val_window.loc[tail_mask[item_idx_val]]),
    "strict_cold": relevant_dict(val_window.loc[strict_cold_mask[item_idx_val]]),
    "warm_tail": relevant_dict(val_window.loc[warm_tail_mask[item_idx_val]]),
}
scope_users = {
    scope: np.asarray(sorted(rel.keys()), dtype=np.int64)
    for scope, rel in relevant_by_scope.items()
}
eval_users = scope_users["overall"]

# Guard against accidental train/validation pair overlap.
_target_u = val_window["u"].to_numpy(dtype=np.int64)
_target_i = val_window["i"].to_numpy(dtype=np.int64)
assert not np.asarray(seen_train[_target_u, _target_i]).ravel().any(), (
    "Some validation target pairs are present in training history."
)

print(f"Users: {n_users:,} | Catalogue movies: {n_items:,}")
print(f"Train interactions: {len(train):,} | Positive train pairs (rating >= {REL}): {len(train_positive):,}")
print(f"Validation rows: {len(val):,} | Evaluated post-2016 positive pairs: {len(val_window):,}")
print(f"content-v1: {X_v1.shape}, nnz={X_v1.nnz:,}")
print(f"TMDB-v2:    {X_tmdb_v2.shape}, nnz={X_tmdb_v2.nnz:,}")
print(f"Tail (<20 positive train pairs): {int(tail_mask.sum()):,} movies")
print(f"Strict cold (0 positive train pairs): {int(strict_cold_mask.sum()):,} movies")
print(f"Warm tail (1–19 positive train pairs): {int(warm_tail_mask.sum()):,} movies")
for _scope, _rel in relevant_by_scope.items():
    print(f"{_scope:12s}: {sum(map(len, _rel.values())):,} relevant pairs, {len(_rel):,} users")
print("Alignment and leakage checks passed. Test split not accessed.")

Users: 20,093 | Catalogue movies: 17,241
Train interactions: 2,919,680 | Positive train pairs (rating >= 4.0): 1,445,945
Validation rows: 137,197 | Evaluated post-2016 positive pairs: 25,406
content-v1: (17241, 31), nnz=53,959
TMDB-v2:    (17241, 91328), nnz=873,444
Tail (<20 positive train pairs): 11,597 movies
Strict cold (0 positive train pairs): 1,095 movies
Warm tail (1–19 positive train pairs): 10,502 movies
overall     : 25,406 relevant pairs, 5,661 users
tail        : 1,585 relevant pairs, 948 users
strict_cold : 52 relevant pairs, 50 users
warm_tail   : 1,533 relevant pairs, 931 users
Alignment and leakage checks passed. Test split not accessed.


## 4 · Build interactions and item feature matrices

### Interactions

WARP needs observed positive user–item interactions. We use training ratings `>= 4.0`, exactly matching the project's definition of a positive. Missing user–movie pairs are treated as unobserved candidates from which the loss samples negatives; they are not written into the interaction matrix as explicit dislike labels.

### Identity versus metadata

- Identity-only: one one-hot item identity feature per movie.
- Metadata-enhanced: the same one-hot identity block plus the selected content representation.
- User features are identity-only in every model; this experiment isolates item metadata.

Before concatenation, each movie's metadata row is L1-normalized. For movies with metadata, the total row weight is divided equally between identity and metadata. This is a fixed, interpretable starting point, not a tuned feature-weight sweep.

In [3]:
# LightFM's implicit-positive interaction matrix.
interactions = sp.coo_matrix(
    (
        np.ones(len(train_positive), dtype=np.float32),
        (
            train_positive["u"].to_numpy(dtype=np.int32),
            train_positive["i"].to_numpy(dtype=np.int32),
        ),
    ),
    shape=(n_users, n_items),
    dtype=np.float32,
)
interactions.sum_duplicates()
interactions.eliminate_zeros()
assert interactions.nnz == len(train_positive)

# Explicit identity features keep the identity-only and hybrid designs conceptually aligned.
I_items = sp.eye(n_items, format="csr", dtype=np.float32)

def make_identity_plus_metadata_features(X_side, metadata_mass=0.5):
    """Return item features with fixed identity/metadata mass and safe empty rows."""
    X_side = X_side.tocsr().astype(np.float32)
    assert X_side.shape[0] == n_items

    X_side = normalize(X_side, norm="l1", axis=1, copy=True).tocsr().astype(np.float32)
    has_metadata = np.asarray(X_side.getnnz(axis=1) > 0).ravel()

    identity_weights = np.where(has_metadata, 1.0 - metadata_mass, 1.0).astype(np.float32)
    identity_block = sp.diags(identity_weights, format="csr") @ I_items
    metadata_block = X_side.multiply(metadata_mass).tocsr()

    features = sp.hstack([identity_block, metadata_block], format="csr", dtype=np.float32)
    row_sums = np.asarray(features.sum(axis=1)).ravel()
    assert np.allclose(row_sums, 1.0, atol=1e-5), (
        f"Item-feature row weights must sum to one; min={row_sums.min()}, max={row_sums.max()}"
    )
    return features

item_features_identity = I_items
item_features_v1 = make_identity_plus_metadata_features(X_v1, metadata_mass=0.5)
item_features_tmdb_v2 = make_identity_plus_metadata_features(X_tmdb_v2, metadata_mass=0.5)

item_feature_sets = {
    "LightFM_identity": item_features_identity,
    "LightFM_content_v1": item_features_v1,
    "LightFM_TMDB_v2": item_features_tmdb_v2,
}

print("Positive interaction matrix:", interactions.shape, "nnz=", f"{interactions.nnz:,}")
for name, feature_matrix in item_feature_sets.items():
    print(
        f"{name:20s} | shape={feature_matrix.shape} | nnz={feature_matrix.nnz:,} | "
        f"item rows with at least one feature={int((feature_matrix.getnnz(axis=1) > 0).sum()):,}"
    )
print("Feature matrices constructed; no feature weights were tuned on validation.")

Positive interaction matrix: (20093, 17241) nnz= 1,445,945
LightFM_identity     | shape=(17241, 17241) | nnz=17,241 | item rows with at least one feature=17,241
LightFM_content_v1   | shape=(17241, 17272) | nnz=71,200 | item rows with at least one feature=17,241
LightFM_TMDB_v2      | shape=(17241, 108569) | nnz=890,685 | item rows with at least one feature=17,241
Feature matrices constructed; no feature weights were tuned on validation.


## 5 · Fit the ALS-only reference on training data

We refit the same implicit ALS setup used in notebook 09, using positive training pairs only. The model is evaluated on the same post-2016 validation window as LightFM. This costs roughly 30–40 seconds in the previous run, but it gives us per-user reference recommendations for paired comparisons.

In [4]:
# Train-only implicit ALS reference, copied from the validated project setup.
def _implicit_half_step(M, Y, lam):
    """One ALS half-step; M stores confidence-minus-one for observed positives."""
    n_rows = M.shape[0]
    k = Y.shape[1]
    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)
    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]
        if start == end:
            continue
        factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]
        A = YtY + (factors.T * confidence_minus_one) @ factors + reg
        b = factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)
    return out


def fit_implicit_als_train_only(df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED):
    d = df.loc[df["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(dtype=np.int64), d["i"].to_numpy(dtype=np.int64)),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)
    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)
        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(f"ALS iteration {iteration:02d}/{n_iter} | elapsed {time.time() - start_time:.1f}s")
    return {"P": P.astype(np.float32), "Q": Q.astype(np.float32)}

als_model = fit_implicit_als_train_only(
    train, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED
)
P_als, Q_als = als_model["P"], als_model["Q"]
item_has_als_factors = np.abs(Q_als).sum(axis=1) > 0
assert P_als.shape == (n_users, 64) and Q_als.shape == (n_items, 64)
assert np.isfinite(P_als).all() and np.isfinite(Q_als).all()
assert int((~item_has_als_factors).sum()) == int(strict_cold_mask.sum())

print("ALS refit complete.")
print("Users without positive history:", int((~u_has_pos).sum()))
print("Items without learned ALS factors:", int((~item_has_als_factors).sum()))

ALS iteration 01/10 | elapsed 4.2s
ALS iteration 02/10 | elapsed 8.4s
ALS iteration 04/10 | elapsed 16.1s
ALS iteration 06/10 | elapsed 24.1s
ALS iteration 08/10 | elapsed 31.7s
ALS iteration 10/10 | elapsed 39.1s
ALS refit complete.
Users without positive history: 34
Items without learned ALS factors: 1095


## 6 · Define and fit the three LightFM models

We use WARP because this experiment is about ranking relevant movies near the top rather than minimizing rating-prediction error. Every model receives the same positive interactions, loss, dimensionality, epoch count, regularization and seed. The metadata models differ only in the item-feature matrix.

These fits can take several minutes. Run each cell once. If a model fails during fitting, stop and share the complete traceback before changing configurations.

In [5]:
def fit_lightfm_candidate(name, item_features):
    start_time = time.time()
    model = LightFM(
        no_components=LIGHTFM_COMPONENTS,
        loss="warp",
        learning_rate=LIGHTFM_LEARNING_RATE,
        item_alpha=LIGHTFM_ITEM_ALPHA,
        user_alpha=LIGHTFM_USER_ALPHA,
        max_sampled=LIGHTFM_MAX_SAMPLED,
        random_state=SEED,
    )
    model.fit(
        interactions,
        item_features=item_features,
        epochs=LIGHTFM_EPOCHS,
        num_threads=NUM_THREADS,
        verbose=True,
    )
    print(f"{name} fit complete in {(time.time() - start_time) / 60:.1f} min")
    print("User embeddings:", model.user_embeddings.shape)
    print("Item feature embeddings:", model.item_embeddings.shape)
    return model

lightfm_models = {}

In [6]:
from scipy.sparse import coo_matrix
from lightfm import LightFM

interactions_test = coo_matrix(
    ([1.0, 1.0, 1.0, 1.0], ([0, 0, 1, 1], [0, 1, 0, 2])),
    shape=(2, 3),
    dtype="float32",
)

model_test = LightFM(no_components=5, loss="warp")
model_test.fit(interactions_test, epochs=2, num_threads=1)

print("Minimal training successful")

Minimal training successful


In [7]:
# 6a · Identity-only LightFM
lightfm_models["LightFM_identity"] = fit_lightfm_candidate(
    "LightFM_identity",
    item_features_identity,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_identity fit complete in 0.4 min
User embeddings: (20093, 32)
Item feature embeddings: (17241, 32)


In [8]:
# 6b · LightFM with the frozen content-v1 metadata (genres + release decade)
lightfm_models["LightFM_content_v1"] = fit_lightfm_candidate(
    "LightFM_content_v1",
    item_features_v1,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_content_v1 fit complete in 0.7 min
User embeddings: (20093, 32)
Item feature embeddings: (17272, 32)


In [9]:
# 6c · LightFM with the TMDB-enriched content-v2 representation
lightfm_models["LightFM_TMDB_v2"] = fit_lightfm_candidate(
    "LightFM_TMDB_v2",
    item_features_tmdb_v2,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
LightFM_TMDB_v2 fit complete in 8.7 min
User embeddings: (20093, 32)
Item feature embeddings: (108569, 32)


## 7 · Verify scoring and prepare full-catalogue scorers

LightFM combines feature embeddings to obtain each item's latent representation and bias. The following scorer uses those representations to compute a batch of user-by-catalogue scores efficiently. It is checked against the official `model.predict` method on a few pairs before full validation scoring.

In [10]:
def make_lightfm_scorer(model, item_features):
    # These products are small dense matrices: (n_items, n_components) plus one bias per item.
    item_vectors = np.asarray(item_features @ model.item_embeddings, dtype=np.float32)
    item_bias = np.asarray(item_features @ model.item_biases).reshape(-1).astype(np.float32)
    user_vectors = np.asarray(model.user_embeddings, dtype=np.float32)
    user_bias = np.asarray(model.user_biases, dtype=np.float32)

    assert item_vectors.shape == (n_items, LIGHTFM_COMPONENTS)
    assert user_vectors.shape == (n_users, LIGHTFM_COMPONENTS)
    assert item_bias.shape == (n_items,) and user_bias.shape == (n_users,)

    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int32)
        scores = user_vectors[ub] @ item_vectors.T
        scores += user_bias[ub, None]
        scores += item_bias[None, :]
        return scores.astype(np.float32, copy=False)

    return score_batch


def make_als_scorer(P, Q):
    def score_batch(user_batch):
        ub = np.asarray(user_batch, dtype=np.int64)
        return (P[ub] @ Q.T).astype(np.float32, copy=False)
    return score_batch

score_functions = {
    "ALS_only": make_als_scorer(P_als, Q_als),
}
for model_name, model in lightfm_models.items():
    score_functions[model_name] = make_lightfm_scorer(
        model, item_feature_sets[model_name]
    )

# Smoke-test direct matrix scoring against LightFM's official predict() on each fitted model.
smoke_user = int(eval_users[0])
smoke_items = np.asarray([0, min(1, n_items - 1), min(17, n_items - 1)], dtype=np.int32)
for model_name, model in lightfm_models.items():
    feature_matrix = item_feature_sets[model_name]
    official = model.predict(
        user_ids=np.full(len(smoke_items), smoke_user, dtype=np.int32),
        item_ids=smoke_items,
        item_features=feature_matrix,
        num_threads=NUM_THREADS,
    )
    direct = score_functions[model_name](np.asarray([smoke_user]))[0, smoke_items]
    assert np.allclose(official, direct, rtol=1e-4, atol=1e-5), (
        f"Scorer mismatch for {model_name}: official={official}, direct={direct}"
    )
    print(f"{model_name}: official-predict scorer check passed")

print("All scorers ready:", list(score_functions))

LightFM_identity: official-predict scorer check passed
LightFM_content_v1: official-predict scorer check passed
LightFM_TMDB_v2: official-predict scorer check passed
All scorers ready: ['ALS_only', 'LightFM_identity', 'LightFM_content_v1', 'LightFM_TMDB_v2']


## 8 · Rank the full catalogue once per model

Recommendations are generated for the users who have at least one relevant target in the overall validation window. The same final recommendation list for each user and tie-breaking seed is used to compute overall, tail, strict-cold and warm-tail metrics.

Rules:

- Remove every item the user rated in training, regardless of rating.
- ALS excludes movies without learned ALS factors, matching the established ALS policy.
- Users without positive training history use the existing popularity fallback for every model.
- LightFM metadata models may score zero-positive-history movies from shared item features; this is one of the behaviours being tested.
- A tiny noise term only resolves nearly exact score ties; it does not constitute score blending.

In [12]:
def popularity_fallback_scores(user_id):
    scores = popularity_scores.copy()
    scores[seen_train[user_id].indices] = -np.inf
    return scores


def top_k_from_scores(scores, user_id, model_name, tie_seed, k=K):
    row = np.asarray(scores, dtype=np.float64).copy()

    # Users with no positive train interactions receive the agreed fallback list.
    if not u_has_pos[user_id]:
        row = popularity_fallback_scores(user_id)
    elif model_name == "ALS_only":
        row[~item_has_als_factors] = -np.inf

    # Never recommend any movie that appeared in the user's training history.
    row[seen_train[user_id].indices] = -np.inf
    eligible = np.isfinite(row)
    if int(eligible.sum()) < k:
        raise RuntimeError(f"Only {int(eligible.sum())} eligible movies for user {user_id}")

    # Seeded per-user tie-break. The same noise vector is used across models for a given
    # user/seed; epsilon is tiny relative to the model score scale.
    tie_rng = np.random.default_rng(np.random.SeedSequence([SEED, int(tie_seed), int(user_id)]))
    tie_noise = tie_rng.random(n_items)
    scale = max(1.0, float(np.max(np.abs(row[eligible]))))
    adjusted = row.copy()
    adjusted[eligible] += tie_noise[eligible] * (1e-8 * scale)

    candidates = np.argpartition(adjusted, -k)[-k:]
    return candidates[np.argsort(adjusted[candidates])[::-1]].astype(np.int32)

seen_sets = {int(u): set(seen_train[int(u)].indices.tolist()) for u in eval_users}
recommendations = {}
start_all = time.time()

for model_name, score_fn in score_functions.items():
    start_model = time.time()
    recs_by_seed = {seed: {} for seed in range(N_TIE_SEEDS)}
    print(f"\nScoring and ranking {model_name} ...")

    for start in range(0, len(eval_users), SCORE_BATCH_SIZE):
        user_batch = eval_users[start:start + SCORE_BATCH_SIZE]
        batch_scores = score_fn(user_batch)
        if batch_scores.shape != (len(user_batch), n_items):
            raise ValueError(
                f"{model_name} returned shape {batch_scores.shape}; "
                f"expected {(len(user_batch), n_items)}"
            )
        if not np.isfinite(batch_scores).all():
            raise ValueError(f"Non-finite scores from {model_name}")

        for row_idx, user_id in enumerate(user_batch):
            raw_scores = batch_scores[row_idx]
            for tie_seed in range(N_TIE_SEEDS):
                recs_by_seed[tie_seed][int(user_id)] = top_k_from_scores(
                    raw_scores, int(user_id), model_name, tie_seed
                )

        processed = min(start + len(user_batch), len(eval_users))
        if processed == len(user_batch) or processed % 512 < SCORE_BATCH_SIZE or processed == len(eval_users):
            print(f"  ranked {processed:,}/{len(eval_users):,} users")

    recommendations[model_name] = recs_by_seed
    print(f"{model_name} completed in {(time.time() - start_model) / 60:.1f} min")
    gc.collect()

print(f"\nAll ranking completed in {(time.time() - start_all) / 60:.1f} min")

# Recommendation-list integrity checks, including no seen items or duplicates.
for model_name, by_seed in recommendations.items():
    for tie_seed, user_recs in by_seed.items():
        assert len(user_recs) == len(eval_users)
        for user_id, slate in user_recs.items():
            assert len(slate) == K and len(set(map(int, slate))) == K
            assert seen_sets[user_id].isdisjoint(map(int, slate)), (model_name, tie_seed, user_id)
    print(f"{model_name}: slate integrity checks passed across {N_TIE_SEEDS} seeds")


Scoring and ranking ALS_only ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
ALS_only completed in 0.4 min

Scoring and ranking LightFM_identity ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,661 users
  ranked 2,560/5,661 users
  ranked 3,072/5,661 users
  ranked 3,584/5,661 users
  ranked 4,096/5,661 users
  ranked 4,608/5,661 users
  ranked 5,120/5,661 users
  ranked 5,632/5,661 users
  ranked 5,661/5,661 users
LightFM_identity completed in 0.4 min

Scoring and ranking LightFM_content_v1 ...
  ranked 64/5,661 users
  ranked 512/5,661 users
  ranked 1,024/5,661 users
  ranked 1,536/5,661 users
  ranked 2,048/5,

## 9 · Calculate user-level metrics for every evaluation scope

We calculate per-user NDCG@10, Recall@10, HitRate@10 and hit count. Metrics are computed separately for overall, tail, strict-cold and warm-tail target sets. For a slice-specific metric, the recommendation candidate pool is still the full catalogue; only the relevant target set is restricted to that slice.

Per-user metrics are averaged over the fixed tie-breaking seeds before bootstrap resampling users.

In [13]:
discounts = 1.0 / np.log2(np.arange(2, K + 2, dtype=np.float64))


def metrics_for_slate(slate, relevant_items):
    relevant_items = set(relevant_items)
    hits = np.asarray([int(item) in relevant_items for item in slate[:K]], dtype=np.float64)
    recall = float(hits.sum() / len(relevant_items)) if relevant_items else 0.0
    idcg_len = min(len(relevant_items), K)
    idcg = float(discounts[:idcg_len].sum()) if idcg_len else 0.0
    ndcg = float((hits * discounts[:len(hits)]).sum() / idcg) if idcg > 0 else 0.0
    return {
        "ndcg@10": ndcg,
        "recall@10": recall,
        "hit_rate@10": float(hits.sum() > 0),
        "hit_count@10": float(hits.sum()),
    }


user_metrics = {model_name: {} for model_name in recommendations}
absolute_metric_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    print(f"Evaluating {scope_name}: {len(users_scope):,} users")

    for model_name in recommendations:
        per_seed_frames = []
        for tie_seed in range(N_TIE_SEEDS):
            values = []
            recs = recommendations[model_name][tie_seed]
            for user_id in users_scope:
                values.append(metrics_for_slate(recs[int(user_id)], rel_dict[int(user_id)]))
            frame = pd.DataFrame(values, index=users_scope)
            frame.index.name = "u"
            per_seed_frames.append(frame)

        averaged = sum(per_seed_frames) / float(N_TIE_SEEDS)
        averaged.index.name = "u"
        user_metrics[model_name][scope_name] = averaged

        for metric in ("ndcg@10", "recall@10", "hit_rate@10", "hit_count@10"):
            absolute_metric_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": float(averaged[metric].mean()),
            })

absolute_metrics = pd.DataFrame(absolute_metric_rows)
print("\nVALIDATION METRIC POINT ESTIMATES")
display(
    absolute_metrics.loc[absolute_metrics["metric"].isin(["ndcg@10", "recall@10", "hit_rate@10"])]
    .pivot_table(index=["scope", "metric"], columns="model", values="mean")
    .round(6)
)

# Sanity check: every model has the same users in every scope.
for scope_name, rel_dict in relevant_by_scope.items():
    for model_name in user_metrics:
        assert user_metrics[model_name][scope_name].index.tolist() == sorted(rel_dict)

Evaluating overall: 5,661 users
Evaluating tail: 948 users
Evaluating strict_cold: 50 users
Evaluating warm_tail: 931 users

VALIDATION METRIC POINT ESTIMATES


model                    ALS_only  LightFM_TMDB_v2  LightFM_content_v1  \
scope       metric                                                       
overall     hit_rate@10  0.302067         0.254902            0.251546   
            ndcg@10      0.081462         0.063340            0.062729   
            recall@10    0.106220         0.083822            0.082748   
strict_cold hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   
tail        hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   
warm_tail   hit_rate@10  0.000000         0.000000            0.000000   
            ndcg@10      0.000000         0.000000            0.000000   
            recall@10    0.000000         0.000000            0.000000   

model                    LightFM_identity  
scope       metric                         
overall     hit_rate@10          0.255255  
            ndcg@10              0.063860  
            recall@10            0.082599  
strict_cold hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000  
tail        hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000  
warm_tail   hit_rate@10          0.000000  
            ndcg@10              0.000000  
            recall@10            0.000000

## 10 · Paired user-level bootstrap confidence intervals

For each comparison we bootstrap the *paired per-user difference*, resampling the same users for both models. This is more informative than comparing two independent intervals because each candidate is evaluated on the same users and targets.

Primary comparison: metadata-enhanced LightFM versus identity-only LightFM on tail View A NDCG@10. Secondary comparisons: each LightFM model versus ALS, plus strict-cold and warm-tail results. The intervals quantify validation uncertainty; they do not undo the fact that multiple model variants were considered.

In [14]:
def paired_bootstrap(delta_values, draw_indices):
    delta_values = np.asarray(delta_values, dtype=np.float64)
    point = float(delta_values.mean())
    boot_means = delta_values[draw_indices].mean(axis=1)
    low, high = np.percentile(boot_means, [2.5, 97.5])
    return point, float(low), float(high)

bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = {}
for scope_name, rel_dict in relevant_by_scope.items():
    n_scope = len(rel_dict)
    bootstrap_indices[scope_name] = bootstrap_rng.integers(
        0, n_scope, size=(N_BOOT, n_scope), endpoint=False
    )

comparisons = [
    ("ALS_only", "LightFM_identity"),
    ("ALS_only", "LightFM_content_v1"),
    ("ALS_only", "LightFM_TMDB_v2"),
    ("LightFM_identity", "LightFM_content_v1"),
    ("LightFM_identity", "LightFM_TMDB_v2"),
]
metrics_to_compare = ("ndcg@10", "recall@10", "hit_rate@10")
paired_rows = []

for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]

    for reference_name, candidate_name in comparisons:
        reference_frame = user_metrics[reference_name][scope_name].loc[users_scope]
        candidate_frame = user_metrics[candidate_name][scope_name].loc[users_scope]

        for metric in metrics_to_compare:
            reference_values = reference_frame[metric].to_numpy(dtype=np.float64)
            candidate_values = candidate_frame[metric].to_numpy(dtype=np.float64)
            delta = candidate_values - reference_values
            point, low, high = paired_bootstrap(delta, draws)
            paired_rows.append({
                "scope": scope_name,
                "reference": reference_name,
                "candidate": candidate_name,
                "metric": metric,
                "users": len(users_scope),
                "reference_mean": float(reference_values.mean()),
                "candidate_mean": float(candidate_values.mean()),
                "paired_delta": point,
                "delta_ci95_low": low,
                "delta_ci95_high": high,
            })

paired_ci_results = pd.DataFrame(paired_rows)
print("PAIRED VALIDATION COMPARISONS")
display(
    paired_ci_results.loc[
        paired_ci_results["metric"].isin(["ndcg@10", "recall@10"])
    ].round(6)
)

primary = paired_ci_results.loc[
    (paired_ci_results["scope"] == "tail")
    & (paired_ci_results["metric"] == "ndcg@10")
    & (paired_ci_results["reference"] == "LightFM_identity")
]
print("\nPRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?")
display(primary.round(6))

PAIRED VALIDATION COMPARISONS


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
0,overall,ALS_only,LightFM_identity,ndcg@10,5661,0.081462,0.063860,-0.017602,-0.021017,-0.014213
1,overall,ALS_only,LightFM_identity,recall@10,5661,0.106220,0.082599,-0.023621,-0.028082,-0.018911
3,overall,ALS_only,LightFM_content_v1,ndcg@10,5661,0.081462,0.062729,-0.018733,-0.022402,-0.015505
4,overall,ALS_only,LightFM_content_v1,recall@10,5661,0.106220,0.082748,-0.023472,-0.028381,-0.018753
6,overall,ALS_only,LightFM_TMDB_v2,ndcg@10,5661,0.081462,0.063340,-0.018122,-0.021618,-0.014545
7,overall,ALS_only,LightFM_TMDB_v2,recall@10,5661,0.106220,0.083822,-0.022398,-0.027485,-0.017533
9,overall,LightFM_identity,LightFM_content_v1,ndcg@10,5661,0.063860,0.062729,-0.001132,-0.003558,0.001309
10,overall,LightFM_identity,LightFM_content_v1,recall@10,5661,0.082599,0.082748,0.000149,-0.003417,0.003487
12,overall,LightFM_identity,LightFM_TMDB_v2,ndcg@10,5661,0.063860,0.063340,-0.000520,-0.002743,0.001482
13,overall,LightFM_identity,LightFM_TMDB_v2,recall@10,5661,0.082599,0.083822,0.001223,-0.002157,0.004525



PRIMARY QUESTION: does item metadata improve LightFM tail View A NDCG@10?


,scope,reference,candidate,metric,users,reference_mean,candidate_mean,paired_delta,delta_ci95_low,delta_ci95_high
24,tail,LightFM_identity,LightFM_content_v1,ndcg@10,948,0.0,0.0,0.0,0.0,0.0
27,tail,LightFM_identity,LightFM_TMDB_v2,ndcg@10,948,0.0,0.0,0.0,0.0,0.0


## 11 · Candidate decision table and reports

The 98% guardrail uses the ALS-only overall View A NDCG@10 **from this exact run**, not a hard-coded historical number. We report both the point-estimate pass/fail and a paired interval for the guardrail margin `G = candidate overall NDCG@10 − 0.98 × ALS overall NDCG@10`.

A LightFM model can still teach us that metadata helps LightFM while failing the overall ALS guardrail. In that case it is a useful experimental result, but it is not selected as the project's replacement recommender under the current rule.

In [15]:
als_overall_frame = user_metrics["ALS_only"]["overall"]
als_overall_ndcg_values = als_overall_frame["ndcg@10"].to_numpy(dtype=np.float64)
als_overall_ndcg = float(als_overall_ndcg_values.mean())
retention_floor = RETENTION_FRACTION * als_overall_ndcg

# Build a concise table with overall, tail, cold and warm-tail point estimates.
decision_rows = []
for model_name in ("ALS_only", "LightFM_identity", "LightFM_content_v1", "LightFM_TMDB_v2"):
    overall = user_metrics[model_name]["overall"]
    tail = user_metrics[model_name]["tail"]
    cold = user_metrics[model_name]["strict_cold"]
    warm = user_metrics[model_name]["warm_tail"]

    overall_ndcg = float(overall["ndcg@10"].mean())
    overall_recall = float(overall["recall@10"].mean())
    guard_low, guard_high = np.nan, np.nan
    guard_point = np.nan
    guard_pass = True if model_name == "ALS_only" else bool(overall_ndcg >= retention_floor)

    if model_name != "ALS_only":
        candidate_overall_values = overall["ndcg@10"].to_numpy(dtype=np.float64)
        guard_margin = candidate_overall_values - RETENTION_FRACTION * als_overall_ndcg_values
        guard_point, guard_low, guard_high = paired_bootstrap(
            guard_margin, bootstrap_indices["overall"]
        )

    decision_rows.append({
        "model": model_name,
        "overall_users": len(overall),
        "overall_ndcg@10": overall_ndcg,
        "overall_recall@10": overall_recall,
        "tail_users": len(tail),
        "tail_ndcg@10": float(tail["ndcg@10"].mean()),
        "tail_recall@10": float(tail["recall@10"].mean()),
        "strict_cold_users": len(cold),
        "strict_cold_ndcg@10": float(cold["ndcg@10"].mean()),
        "strict_cold_recall@10": float(cold["recall@10"].mean()),
        "warm_tail_users": len(warm),
        "warm_tail_ndcg@10": float(warm["ndcg@10"].mean()),
        "warm_tail_recall@10": float(warm["recall@10"].mean()),
        "als_overall_ndcg@10": als_overall_ndcg,
        "overall_retention_floor": retention_floor,
        "overall_guardrail_pass_point_estimate": guard_pass,
        "guardrail_margin_G": guard_point,
        "guardrail_G_ci95_low": guard_low,
        "guardrail_G_ci95_high": guard_high,
    })

decision_table = pd.DataFrame(decision_rows).sort_values(
    ["overall_guardrail_pass_point_estimate", "tail_ndcg@10"],
    ascending=[False, False],
).reset_index(drop=True)

print("CANDIDATE DECISION TABLE")
display(decision_table.round(6))

# A small config artifact for reproducibility.
run_config = {
    "notebook": "10_lightfm_hybrid.ipynb",
    "dataset_subset": NAME,
    "positive_rating_threshold": REL,
    "validation_window_start_utc": "2016-01-01T00:00:00Z",
    "tail_definition": "fewer than 20 positive train interactions",
    "strict_cold_definition": "zero positive train interactions",
    "warm_tail_definition": "1-19 positive train interactions",
    "test_split_used": False,
    "lightfm": {
        "loss": "warp",
        "components": LIGHTFM_COMPONENTS,
        "epochs": LIGHTFM_EPOCHS,
        "learning_rate": LIGHTFM_LEARNING_RATE,
        "item_alpha": LIGHTFM_ITEM_ALPHA,
        "user_alpha": LIGHTFM_USER_ALPHA,
        "max_sampled": LIGHTFM_MAX_SAMPLED,
        "num_threads": NUM_THREADS,
        "metadata_weighting": "for rows with metadata: identity feature mass 0.5, normalized metadata feature mass 0.5; empty metadata rows use identity mass 1.0",
    },
    "bootstrap_resamples": N_BOOT,
    "tie_breaking_seeds": N_TIE_SEEDS,
    "random_seed": SEED,
    "als_overall_view_a_ndcg_at_10": als_overall_ndcg,
    "retention_fraction": RETENTION_FRACTION,
    "retention_floor": retention_floor,
}

REPORT_DIR = _PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save per-model metrics with absolute user-bootstrap CIs.
absolute_rows = []
for scope_name, rel_dict in relevant_by_scope.items():
    users_scope = np.asarray(sorted(rel_dict), dtype=np.int64)
    draws = bootstrap_indices[scope_name]
    for model_name in user_metrics:
        frame = user_metrics[model_name][scope_name].loc[users_scope]
        for metric in metrics_to_compare:
            values = frame[metric].to_numpy(dtype=np.float64)
            point, low, high = paired_bootstrap(values, draws)
            absolute_rows.append({
                "model": model_name,
                "scope": scope_name,
                "metric": metric,
                "users": len(users_scope),
                "mean": point,
                "ci95_low": low,
                "ci95_high": high,
            })
absolute_ci_table = pd.DataFrame(absolute_rows)

metrics_path = REPORT_DIR / "lightfm_validation_metrics.csv"
paired_path = REPORT_DIR / "lightfm_paired_validation_ci.csv"
decision_path = REPORT_DIR / "lightfm_candidate_decision.csv"
config_path = REPORT_DIR / "lightfm_run_config.json"
user_metrics_path = REPORT_DIR / "lightfm_user_metrics.csv"

absolute_ci_table.to_csv(metrics_path, index=False)
paired_ci_results.to_csv(paired_path, index=False)
decision_table.to_csv(decision_path, index=False)
(Path(config_path)).write_text(json.dumps(run_config, indent=2) + "\n", encoding="utf-8")

user_metric_export_rows = []
for model_name, scopes in user_metrics.items():
    for scope_name, frame in scopes.items():
        export = frame.reset_index().copy()
        export.insert(0, "scope", scope_name)
        export.insert(0, "model", model_name)
        user_metric_export_rows.append(export)
pd.concat(user_metric_export_rows, ignore_index=True).to_csv(user_metrics_path, index=False)

print("\nSaved reports:")
for path in (metrics_path, paired_path, decision_path, config_path, user_metrics_path):
    print(path)

print("\nOVERALL GUARDRAIL")
print(f"ALS overall View A NDCG@10: {als_overall_ndcg:.6f}")
print(f"98% retention floor:        {retention_floor:.6f}")
eligible = decision_table.loc[
    (decision_table["model"] != "ALS_only")
    & decision_table["overall_guardrail_pass_point_estimate"]
]
if len(eligible):
    selected = eligible.sort_values("tail_ndcg@10", ascending=False).iloc[0]
    print("Best LightFM candidate passing point-estimate guardrail:", selected["model"])
    print(f"Overall NDCG@10: {selected['overall_ndcg@10']:.6f} | Tail NDCG@10: {selected['tail_ndcg@10']:.6f}")
    print(
        "Guardrail G 95% CI: "
        f"[{selected['guardrail_G_ci95_low']:+.6f}, {selected['guardrail_G_ci95_high']:+.6f}]"
    )
else:
    print("No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.")

print("Test split used: No")

CANDIDATE DECISION TABLE


,model,overall_users,overall_ndcg@10,overall_recall@10,tail_users,tail_ndcg@10,tail_recall@10,strict_cold_users,strict_cold_ndcg@10,strict_cold_recall@10,warm_tail_users,warm_tail_ndcg@10,warm_tail_recall@10,als_overall_ndcg@10,overall_retention_floor,overall_guardrail_pass_point_estimate,guardrail_margin_G,guardrail_G_ci95_low,guardrail_G_ci95_high
0,ALS_only,5661,0.081462,0.106220,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,True,NaN,NaN,NaN
1,LightFM_identity,5661,0.063860,0.082599,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.015973,-0.019339,-0.012606
2,LightFM_content_v1,5661,0.062729,0.082748,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.017104,-0.020709,-0.013898
3,LightFM_TMDB_v2,5661,0.063340,0.083822,948,0.0,0.0,50,0.0,0.0,931,0.0,0.0,0.081462,0.079833,False,-0.016492,-0.019969,-0.012963



Saved reports:
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_validation_metrics.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_paired_validation_ci.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_candidate_decision.csv
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_run_config.json
/mnt/c/Users/Lenovo/Documents/movie-recommender/reports/lightfm_user_metrics.csv

OVERALL GUARDRAIL
ALS overall View A NDCG@10: 0.081462
98% retention floor:        0.079833
No LightFM candidate passes the overall point-estimate guardrail. Keep ALS as the project reference.
Test split used: No


## 12 · How to interpret the outcome

Use `lightfm_candidate_decision.csv` for the high-level results, `lightfm_paired_validation_ci.csv` for paired model differences, and `lightfm_validation_metrics.csv` for absolute means and confidence intervals.

Interpret the results in this order:

1. **Did metadata help LightFM?** Check `LightFM_content_v1` and `LightFM_TMDB_v2` versus `LightFM_identity` on tail View A NDCG@10. A positive point estimate with an interval crossing zero is inconclusive.
2. **Did LightFM beat the existing ALS reference?** Look at overall and tail metrics plus paired intervals against `ALS_only`.
3. **Does it meet the project guardrail?** It must meet the 98% overall NDCG point-estimate floor to be considered as a replacement under the current selection rule.
4. **What about true cold-start?** Strict-cold validation has only a small number of relevant users/targets; treat that estimate as noisy and inspect its paired interval and hit counts.
5. **Don't tune on the test set.** If one LightFM candidate is promising, any limited follow-up tuning stays within validation. The test split is reserved for the final frozen comparison.

This is the first controlled LightFM run, not a definitive hyperparameter search. If it does not help, report that honestly; don't automatically increase metadata weight or epochs until a favourable validation result appears.